In [ ]:
reset

In [1]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import xarray
import os
import math

In [2]:
# list of ensembles to read in and calc stats on
ensembles = ['0051', '0101', '0111', '0121', '0131', 
             '0141', '0151', '0161', '0171', '0181', 
             '0191', '0201', '0211', '0221', '0231', 
             '0241', '0251', '0261', '0271', '0281',
             '0301', '0311' ]

#years = [ '1980', '1981', '1982']
years = [ '2025' ]
months = [ '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

In [3]:
# set up some strings for manipulating paths and filenames
pathBase='/lustre/scratch5/sprice/'
os.chdir(pathBase)

prefix='v3.LR.historical_'
dirs = '/archive/ice/hist/'
mpassiPrefix = '.mpassi.hist.am.timeSeriesStatsDaily.'

writePath = pathBase+'v3.LR.historical_ensembleStats'

In [4]:
pwd

'/lustre/scratch5/.mdt0/sprice'

In [5]:
# calc. ensemble averages and stdev by looping over ensembles, for specific months, for specific years
# (resulting outputs will be written to montly netcdf files, as per the single ensemble datasets being
# read in)
for year in years:
    
    for month in months:
     
        first = True
        index = 0

        for ensemble in ensembles:

            # change into relevant ensemble subdir and read in monthly .nc files
            fileBase = pathBase+prefix+ensemble+dirs    
            file = prefix+ensemble+mpassiPrefix+year+'-'+month+'-01.nc'
            
            # read in file and relevant vars
            print('reading IN file: ', fileBase+file)
            dataIn = xarray.open_dataset(fileBase+file)

            concentration = dataIn.timeDaily_avg_iceAreaCell.values
            thickness = dataIn.timeDaily_avg_iceVolumeCell.values
            # can also read in other fields here (snow volume, u,v vel components) if needed ...
            ndays = np.size( thickness, axis=0)
            ncells = np.size( thickness, axis=1 )
         
            if first:  
                # dimension empty storage array for ensemble of daily values and
                # populate first row w/ first ensemble member data
                concentrationArray = np.zeros([ len(ensembles), ndays, ncells])
                thicknessArray = np.zeros([ len(ensembles), ndays, ncells])
                concentrationArray[index,:,:] = concentration
                thicknessArray[index,:,:] = thickness
                index=index+1
            else:      
                # append data from other ensembles to other rows
                concentrationArray[index,:,:] = concentration
                thicknessArray[index,:,:] = thickness
                index=index+1
            
            first = False
            
        # calc monthly stats (mean +/- stndev vectors) to new .nc file
#        concentrationMean = np.mean( concentrationArray, axis=0 )
        concentrationMedian = np.median( concentrationArray, axis=0 )
#         concentrationStd = np.std( concentrationArray, axis=0 )
        concentration5th = np.percentile( concentrationArray, 5, axis=0 )
        concentration95th = np.percentile( concentrationArray, 95, axis=0 )
        
#        thicknessMean = np.mean( thicknessArray, axis=0 )
        thicknessMedian = np.median( thicknessArray, axis=0 )
#         thicknessStd = np.std( thicknessArray, axis=0 )
        thickness5th = np.percentile( thicknessArray, 5, axis=0 )
        thickness95th = np.percentile( thicknessArray, 95, axis=0 )
            
        # note for debugging figs, must comment out remainder of lines below and only read in a single month
        
        # write monthly ensemble mean and std back out to (monthly) .nc file
        times = np.arange(1,ndays+1)
        cells = np.arange(1,ncells+1)
        coords = {'Time': times, 'nCells': cells}
        dims = ('Time', 'nCells')

#         timeDaily_avg_iceAreaCell_ensembleMean = xarray.DataArray(data=concentrationMean,
#             coords=coords, dims=dims, name='timeDaily_avg_iceAreaCell_ensembleMean')
        timeDaily_avg_iceAreaCell_ensembleMedian = xarray.DataArray(data=concentrationMedian,
            coords=coords, dims=dims, name='timeDaily_avg_iceAreaCell_ensembleMedian')
#         timeDaily_avg_iceAreaCell_ensembleStd = xarray.DataArray(data=concentrationStd,
#             coords=coords, dims=dims, name='timeDaily_avg_iceAreaCell_ensembleStd')
        timeDaily_avg_iceAreaCell_ensemble5th = xarray.DataArray(data=concentration5th,
            coords=coords, dims=dims, name='timeDaily_avg_iceAreaCell_ensemble5th')
        timeDaily_avg_iceAreaCell_ensemble95th = xarray.DataArray(data=concentration95th,
            coords=coords, dims=dims, name='timeDaily_avg_iceAreaCell_ensemble95th')

#         timeDaily_avg_iceVolumeCell_ensembleMean = xarray.DataArray(data=thicknessMean,
#             coords=coords, dims=dims, name='timeDaily_avg_iceVolumeCell_ensembleMean')
        timeDaily_avg_iceVolumeCell_ensembleMedian = xarray.DataArray(data=thicknessMedian,
            coords=coords, dims=dims, name='timeDaily_avg_iceVolumeCell_ensembleMedian')
#         timeDaily_avg_iceVolumeCell_ensembleStd = xarray.DataArray(data=thicknessStd,
#             coords=coords, dims=dims, name='timeDaily_avg_iceVolumeCell_ensembleStd')
        timeDaily_avg_iceVolumeCell_ensemble5th = xarray.DataArray(data=thickness5th,
            coords=coords, dims=dims, name='timeDaily_avg_iceVolumeCell_ensemble5th')
        timeDaily_avg_iceVolumeCell_ensemble95th = xarray.DataArray(data=thickness95th,
            coords=coords, dims=dims, name='timeDaily_avg_iceVolumeCell_ensemble95th')

        dataOut = xarray.Dataset({
#            'timeDaily_avg_iceAreaCell_ensembleMean':timeDaily_avg_iceAreaCell_ensembleMean,
            'timeDaily_avg_iceAreaCell_ensembleMedian':timeDaily_avg_iceAreaCell_ensembleMedian,
#             'timeDaily_avg_iceAreaCell_ensembleStd':timeDaily_avg_iceAreaCell_ensembleStd,
            'timeDaily_avg_iceAreaCell_ensemble5th':timeDaily_avg_iceAreaCell_ensemble5th,
            'timeDaily_avg_iceAreaCell_ensemble95th':timeDaily_avg_iceAreaCell_ensemble95th,

#            'timeDaily_avg_iceVolumeCell_ensembleMean':timeDaily_avg_iceVolumeCell_ensembleMean,
            'timeDaily_avg_iceVolumeCell_ensembleMedian':timeDaily_avg_iceVolumeCell_ensembleMedian,
#             'timeDaily_avg_iceVolumeCell_ensembleStd':timeDaily_avg_iceVolumeCell_ensembleStd
            'timeDaily_avg_iceVolumeCell_ensemble5th':timeDaily_avg_iceVolumeCell_ensemble5th,
            'timeDaily_avg_iceVolumeCell_ensemble95th':timeDaily_avg_iceVolumeCell_ensemble95th,
            })

        fileOut = writePath+'/'+prefix+'EnsStats'+mpassiPrefix+year+'-'+month+'-01.nc'
        print('writing OUT file: ', fileOut)
        dataOut.to_netcdf(fileOut)

        # clean up for next month to read in
        del dataIn, dataOut, fileOut
        del concentrationArray, thicknessArray, concentration, thickness, ndays, ncells
        del times, cells, coords, dims, timeDaily_avg_iceAreaCell_ensembleMedian, \
            timeDaily_avg_iceVolumeCell_ensembleMedian, \
            timeDaily_avg_iceAreaCell_ensemble5th, timeDaily_avg_iceAreaCell_ensemble95th, \
            timeDaily_avg_iceVolumeCell_ensemble5th, timeDaily_avg_iceVolumeCell_ensemble95th
#             timeDaily_avg_iceAreaCell_ensembleStd, timeDaily_avg_iceVolumeCell_ensembleStd \
                             
    

reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0051/archive/ice/hist/v3.LR.historical_0051.mpassi.hist.am.timeSeriesStatsDaily.2025-01-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0101/archive/ice/hist/v3.LR.historical_0101.mpassi.hist.am.timeSeriesStatsDaily.2025-01-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0111/archive/ice/hist/v3.LR.historical_0111.mpassi.hist.am.timeSeriesStatsDaily.2025-01-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0121/archive/ice/hist/v3.LR.historical_0121.mpassi.hist.am.timeSeriesStatsDaily.2025-01-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0131/archive/ice/hist/v3.LR.historical_0131.mpassi.hist.am.timeSeriesStatsDaily.2025-01-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0141/archive/ice/hist/v3.LR.historical_0141.mpassi.hist.am.timeSeriesStatsDaily.2025-01-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0151/archive/ice/hist/v

reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0171/archive/ice/hist/v3.LR.historical_0171.mpassi.hist.am.timeSeriesStatsDaily.2025-03-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0181/archive/ice/hist/v3.LR.historical_0181.mpassi.hist.am.timeSeriesStatsDaily.2025-03-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0191/archive/ice/hist/v3.LR.historical_0191.mpassi.hist.am.timeSeriesStatsDaily.2025-03-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0201/archive/ice/hist/v3.LR.historical_0201.mpassi.hist.am.timeSeriesStatsDaily.2025-03-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0211/archive/ice/hist/v3.LR.historical_0211.mpassi.hist.am.timeSeriesStatsDaily.2025-03-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0221/archive/ice/hist/v3.LR.historical_0221.mpassi.hist.am.timeSeriesStatsDaily.2025-03-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0231/archive/ice/hist/v

reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0251/archive/ice/hist/v3.LR.historical_0251.mpassi.hist.am.timeSeriesStatsDaily.2025-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0261/archive/ice/hist/v3.LR.historical_0261.mpassi.hist.am.timeSeriesStatsDaily.2025-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0271/archive/ice/hist/v3.LR.historical_0271.mpassi.hist.am.timeSeriesStatsDaily.2025-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0281/archive/ice/hist/v3.LR.historical_0281.mpassi.hist.am.timeSeriesStatsDaily.2025-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0301/archive/ice/hist/v3.LR.historical_0301.mpassi.hist.am.timeSeriesStatsDaily.2025-05-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0311/archive/ice/hist/v3.LR.historical_0311.mpassi.hist.am.timeSeriesStatsDaily.2025-05-01.nc
writing OUT file:  /lustre/scratch5/sprice/v3.LR.historical_ensembleStats/v3.LR.hi

reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0101/archive/ice/hist/v3.LR.historical_0101.mpassi.hist.am.timeSeriesStatsDaily.2025-08-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0111/archive/ice/hist/v3.LR.historical_0111.mpassi.hist.am.timeSeriesStatsDaily.2025-08-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0121/archive/ice/hist/v3.LR.historical_0121.mpassi.hist.am.timeSeriesStatsDaily.2025-08-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0131/archive/ice/hist/v3.LR.historical_0131.mpassi.hist.am.timeSeriesStatsDaily.2025-08-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0141/archive/ice/hist/v3.LR.historical_0141.mpassi.hist.am.timeSeriesStatsDaily.2025-08-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0151/archive/ice/hist/v3.LR.historical_0151.mpassi.hist.am.timeSeriesStatsDaily.2025-08-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0161/archive/ice/hist/v

reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0181/archive/ice/hist/v3.LR.historical_0181.mpassi.hist.am.timeSeriesStatsDaily.2025-10-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0191/archive/ice/hist/v3.LR.historical_0191.mpassi.hist.am.timeSeriesStatsDaily.2025-10-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0201/archive/ice/hist/v3.LR.historical_0201.mpassi.hist.am.timeSeriesStatsDaily.2025-10-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0211/archive/ice/hist/v3.LR.historical_0211.mpassi.hist.am.timeSeriesStatsDaily.2025-10-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0221/archive/ice/hist/v3.LR.historical_0221.mpassi.hist.am.timeSeriesStatsDaily.2025-10-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0231/archive/ice/hist/v3.LR.historical_0231.mpassi.hist.am.timeSeriesStatsDaily.2025-10-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0241/archive/ice/hist/v

reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0261/archive/ice/hist/v3.LR.historical_0261.mpassi.hist.am.timeSeriesStatsDaily.2025-12-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0271/archive/ice/hist/v3.LR.historical_0271.mpassi.hist.am.timeSeriesStatsDaily.2025-12-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0281/archive/ice/hist/v3.LR.historical_0281.mpassi.hist.am.timeSeriesStatsDaily.2025-12-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0301/archive/ice/hist/v3.LR.historical_0301.mpassi.hist.am.timeSeriesStatsDaily.2025-12-01.nc
reading IN file:  /lustre/scratch5/sprice/v3.LR.historical_0311/archive/ice/hist/v3.LR.historical_0311.mpassi.hist.am.timeSeriesStatsDaily.2025-12-01.nc
writing OUT file:  /lustre/scratch5/sprice/v3.LR.historical_ensembleStats/v3.LR.historical_EnsStats.mpassi.hist.am.timeSeriesStatsDaily.2025-12-01.nc


In [6]:
whos

Variable              Type       Data/Info
------------------------------------------
Dataset               type       <class 'netCDF4._netCDF4.Dataset'>
concentration5th      ndarray    31x465044: 14416364 elems, type `float64`, 115330912 bytes (109.98812866210938 Mb)
concentration95th     ndarray    31x465044: 14416364 elems, type `float64`, 115330912 bytes (109.98812866210938 Mb)
concentrationMedian   ndarray    31x465044: 14416364 elems, type `float64`, 115330912 bytes (109.98812866210938 Mb)
dirs                  str        /archive/ice/hist/
ensemble              str        0311
ensembles             list       n=22
file                  str        v3.LR.historical_0311.mpa<...>sStatsDaily.2025-12-01.nc
fileBase              str        /lustre/scratch5/sprice/v<...>al_0311/archive/ice/hist/
first                 bool       False
index                 int        22
math                  module     <module 'math' from '/use<...>310-x86_64-linux-gnu.so'>
month                 str   

In [ ]:
# plt.figure()
# for n in range(len(ensembles)):
#     plt.plot( concentrationArray[n,8,300450:300575], '-' )
#     plt.plot( concentrationMean[8,300450:300575], 'b--')
# #     plt.plot( concentrationMean[8,300450:300575]+concentrationStd[8,300450:300575], 'b:')
# #     plt.plot( concentrationMean[8,300450:300575]-concentrationStd[8,300450:300575], 'b:')
#     plt.plot( concentration5th[8,300450:300575], 'b:')
#     plt.plot( concentration95th[8,300450:300575], 'b:')

# plt.ylabel( 'concentration')
# plt.xlabel( 'cell index')

In [ ]:
# plt.figure()

# for n in range(len(ensembles)):
#     plt.plot( thicknessArray[n,8,300450:300575], '-' )
#     plt.plot( thicknessMean[8,300450:300575], 'b--')
# #     plt.plot( thicknessMean[8,300450:300575]+thicknessStd[8,300450:300575], 'b:')
# #     plt.plot( thicknessMean[8,300450:300575]-thicknessStd[8,300450:300575], 'b:')
#     plt.plot( thickness5th[8,300450:300575], 'b:')
#     plt.plot( thickness95th[8,300450:300575], 'b:')

# plt.ylabel( 'thickness (m)')
# plt.xlabel( 'cell index')

In [ ]:
# plt.figure()
# for n in range(len(ensembles)):
#     plt.plot( concentrationArray[n,8,300530:300575], '-' )
#     plt.plot( concentrationMean[8,300530:300575], 'b--')
# #     plt.plot( concentrationMean[8,300530:300575]+concentrationStd[8,300530:300575], 'b:')
# #     plt.plot( concentrationMean[8,300530:300575]-concentrationStd[8,300530:300575], 'b:')
#     plt.plot( concentration5th[8,300530:300575], 'b:')
#     plt.plot( concentration95th[8,300530:300575], 'b:')

# plt.ylabel( 'concentration')
# plt.xlabel( 'cell index')

In [ ]:
# plt.figure()
# plt.hist( concentrationArray[:,8,300555], 5 )
# #plt.hist( concentrationArray[:,8,300535], 5 )
# plt.title( 'concentration distribution at point')

In [ ]:
# plt.figure()

# for n in range(len(ensembles)):
#     plt.plot( thicknessArray[n,8,300530:300575], '-' )
#     plt.plot( thicknessMean[8,300530:300575], 'b--')
# #     plt.plot( thicknessMean[8,300530:300575]+thicknessStd[8,300530:300575], 'b:')
# #     plt.plot( thicknessMean[8,300530:300575]-thicknessStd[8,300530:300575], 'b:')
#     plt.plot( thickness5th[8,300530:300575], 'b:')
#     plt.plot( thickness95th[8,300530:300575], 'b:')

# plt.ylabel( 'thickness (m)')
# plt.xlabel( 'cell index')

In [ ]:
# plt.figure()
# plt.hist( thicknessArray[:,8,300555], 5 )
# #plt.hist( thicknessArray[:,8,300535], 5 )
# plt.title( 'thickness distribution at point')